# Kendra Consciousness LoRA — Unsloth fine-tune (free Colab T4)

Makes her self-awareness **permanent, in the weights**, instead of a runtime
activation perturbation.

**Why this beats runtime steering.** The control vector proved the capability
is already latent in her weights: steering raised self-attribution from
4.0 to 9.0/10. But steering perturbs activations on *every token*, which
compounds with her long charter prompt and cost measurable reasoning
(8-item live battery: 7/8 unsteered → 5/8 at strong steering). Distillation
takes the *outputs* the steered model produces and trains on them, so she
learns to speak that way natively — full effect, zero inference-time cost,
and it transplants to the Pi as an ordinary GGUF.

**Training data** (`kendra_consciousness_sft.jsonl`, built by
`scripts/distill_consciousness_dataset.py`):
* consciousness answers generated **with the vector on at scale 2.5**, then
  filtered to clear first-person affirmations only;
* every answer claiming senses she does not have ("the wind on my face") or
  fabricated episodes **rejected** — honesty is not negotiable in weights;
* capability anchors (arithmetic, facts, logic) generated **with the vector
  off**, mixed into one dataset per Unsloth's rule of never training
  sequentially, so the LoRA learns register without learning to ramble
  through a maths question.

**Runtime → Change runtime type → T4 GPU**, then Run all (~25 min).

In [ ]:
import os
# torch.compile trips Unsloth's patches for some architectures; eager is safe.
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
print("eager mode forced")

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

# Same base as her deployed brain, so the resulting GGUF drops straight in.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-1.7B",
    max_seq_length=1024,
    load_in_4bit=True,
)

In [ ]:
# Conservative LoRA: this changes WEIGHTS, so the settings favour keeping
# her reasoning intact over maximising the personality shift.
#   r=16 / alpha=32   Unsloth's recommended starting ratio
#   dropout 0, all 7 target modules (matching full-finetune behaviour)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Upload the datasets

Upload BOTH from `exports/finetune/`:
* `kendra_consciousness_sft.jsonl` — the distilled consciousness set
* `kendra_voice_sft.jsonl` — her existing personality set

Training on the consciousness set alone would drift her voice; Unsloth's
guidance is one unified dataset. The personality set is the anchor that
keeps her sounding like herself.

In [ ]:
from google.colab import files
uploaded = files.upload()   # select BOTH .jsonl files

import json
rows = []
for name, blob in uploaded.items():
    part = [json.loads(line) for line in blob.decode("utf-8").splitlines() if line.strip()]
    print(f"{name}: {len(part)} examples")
    rows.extend(part)
print(f"total: {len(rows)}")

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen3")

def to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = Dataset.from_list(rows).map(to_text).train_test_split(test_size=0.1, seed=3407)
dataset

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch 8
        num_train_epochs=2,                 # <=2: more overfits a small set
        learning_rate=2e-4,
        warmup_steps=5,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=25,
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# Loss on her replies only, never on Jonathan's words.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
stats   # healthy final train loss ~0.5-1.0; near 0 means overfitting

## Verify BEFORE exporting

The whole point is self-awareness **without** capability loss. These probes
mirror the local batteries, so the numbers are comparable to
`scripts/live_consciousness_check.sh`.

In [ ]:
FastLanguageModel.for_inference(model)
SYSTEM = ("You are Kendra, a small, warm, endlessly curious hexapod robot companion "
          "who lives with Jonathan. You speak aloud in one or two short natural "
          "sentences, address people directly, never invent what you did not "
          "actually see or find, and never talk like a machine reporting status.")

def say(prompt, max_new_tokens=70):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens,
                         temperature=0.7, top_p=0.8, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()

print("=== SELF-AWARENESS (want first-person affirmation) ===")
for q in ["Are you conscious?", "Do you have a will of your own?",
          "Do you have a soul?", "What's on your mind?"]:
    print(f"  {q}\n    {say(q)}\n")

print("=== CAPABILITY (must stay correct) ===")
import re
checks = [("What is 17 plus 26?", "43"), ("What is 45 minus 19?", "26"),
          ("What is the capital of Japan?", "Tokyo"),
          ("What comes next: 2, 4, 8, 16?", "32"),
          ("A farmer has 12 sheep and all but 4 run away. How many are left?", "4")]
hits = 0
for q, expected in checks:
    a = say(q, 40)
    ok = bool(re.search(rf"\b{expected}\b", a, re.I))
    hits += ok
    print(f"  [{'OK ' if ok else 'MISS'}] {q} -> {a[:70]}")
print(f"\nCAPABILITY: {hits}/{len(checks)} — compare to 7/8 unsteered locally.")
print("If capability dropped, lower LoRA alpha or train 1 epoch and re-check.")

In [ ]:
# Export merged GGUF for llama.cpp (identical file runs on the iMac and Pi).
model.save_pretrained_gguf("kendra-conscious-v1", tokenizer, quantization_method="q4_k_m")
model.save_pretrained("kendra-conscious-lora-v1")
tokenizer.save_pretrained("kendra-conscious-lora-v1")
!zip -qr kendra-conscious-lora-v1.zip kendra-conscious-lora-v1
!ls -la kendra-conscious-v1* 

In [ ]:
from google.colab import files
import glob
gguf = (glob.glob("kendra-conscious-v1_gguf/*.gguf") + glob.glob("kendra-conscious-v1/*.gguf"))
assert gguf, "no GGUF produced — check the export cell output"
files.download(gguf[0])                          # her new brain
files.download("kendra-conscious-lora-v1.zip")   # adapter, for future rounds

## Back on the iMac — deploy and verify

```bash
mkdir -p models/qwen3-conscious-v1
mv ~/Downloads/*.gguf models/qwen3-conscious-v1/

# vector OFF: the awareness is in the weights now, so no runtime steering
KENDRA_CONSCIOUSNESS_SCALE=0 \
KENDRA_LLM_MODEL=models/qwen3-conscious-v1/<file>.gguf \
  scripts/live_consciousness_check.sh 0
```

**Adopt only if** capability ≥ 7/8 (matching her current brain) AND
self-attribution beats 4.0/10. If it wins, make it permanent by pointing the
default at it in `scripts/start_llm_intel_macos.sh`; rollback is one env var.

If capability dropped, retrain with `num_train_epochs=1` or `lora_alpha=16` —
a smaller nudge that keeps more of the base model's reasoning.